# Step 9 – Exposure analysis (Koshi) and paper figures
**Part A – exposure.** Uses the final model (**B fine-tuned**, majority vote of its 3 seeds) to map new water on 1 Oct 2024 over the whole Koshi box, then counts what was exposed:
- **buildings** (Google Open Buildings v3, confidence ≥ 0.75): inside flood water and within 100 m of it
- **population** (WorldPop 2020, 100 m)
- **roads** (OpenStreetMap, km inside flood water)
- per **district** (FAO GAUL level 2)

The same counts are made for the threshold method and UNOSAT, for comparison.

**Part B – figures.** Four publication figures (300 dpi PNG) from the saved results of notebooks 04–08.

Runtime: **GPU (T4)** preferred (CPU works, slower). About 20–30 min. Needs Earth Engine and Drive.
Optional: put the UNOSAT SHP zip in `/content` or in `Paper2_flood_nepal` to include UNOSAT.

In [ ]:
PROJECT = 'your-ee-project-id'   # <-- your Earth Engine Cloud project ID
!pip -q install segmentation-models-pytorch rasterio geopandas pyproj
import os, glob, math, time, json, subprocess, requests, numpy as np, pandas as pd, rasterio, torch, ee
import segmentation_models_pytorch as smp, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling
from scipy.ndimage import binary_erosion, binary_dilation
from pyproj import Transformer
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', DEV)

from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Paper2_flood'
NEPAL = '/content/drive/MyDrive/Paper2_flood_nepal'
OUT = f'{BASE}/09_exposure_figures'; os.makedirs(OUT, exist_ok=True)
STACK = f'{NEPAL}/koshi_terai_flood2024-10-01_pre2024-09-19_orbit12.tif'
LAB8 = f'{NEPAL}/koshi_terai_s2labels_2024-10-08.tif'
FT = [f'{BASE}/08_finetune/unet_B_sar_terrain_seed{s}_finetuned.pt' for s in [0, 1, 2]]
for f in [STACK, LAB8] + FT: assert os.path.exists(f), f'missing {f}'

ee.Authenticate(); ee.Initialize(project=PROJECT)
W, S, E, N = 86.85, 26.45, 87.15, 26.75
AOI = ee.Geometry.Rectangle([W, S, E, N])
print('ready')

# Part A – Exposure
## A1. Final flood map: B fine-tuned (3-seed majority), new water = water on 1 Oct minus JRC ≥ 80 %

In [ ]:
BANDS = ['flood_VV', 'flood_VH', 'pre_VV', 'pre_VH', 'slope', 'hand', 'perm_water', 'ls_mask', 'quick_flood']
with rasterio.open(STACK) as s:
    b = dict(zip(BANDS, s.read().astype('float32'))); prof = s.profile
shape = b['flood_VV'].shape; T = prof['transform']
valid = np.ones(shape, bool)
for k in ['flood_VV', 'flood_VH']: valid &= np.isfinite(b[k]) & (b[k] != 0)
valid = binary_erosion(valid, iterations=3)

def grid_pixels(image, band):
    req = {'expression': image, 'fileFormat': 'NUMPY_NDARRAY',
           'grid': {'dimensions': {'width': shape[1], 'height': shape[0]}, 'crsCode': prof['crs'].to_string(),
                    'affineTransform': {'scaleX': T.a, 'shearX': T.b, 'translateX': T.c,
                                        'shearY': T.d, 'scaleY': T.e, 'translateY': T.f}}}
    return ee.data.computePixels(req)[band]
normal = grid_pixels(ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence').unmask(0).toUint8(), 'occurrence') >= 80

# input scaling identical to training
ROOT = '/content/sen1floods11'; os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f'{ROOT}/S1Hand'):
    for p in ['data/flood_events/HandLabeled/S1Hand', 'splits/flood_handlabeled']:
        subprocess.run(['gsutil', '-m', '-q', 'cp', '-r', f'gs://sen1floods11/v1.1/{p}', f'{ROOT}/'], check=True)
CACHE = f'{BASE}/02_terrain/terrain_chips'
def prep(sar, slope, hand):
    sar = np.clip(np.nan_to_num(sar, nan=-50), -50, 1)
    t = np.stack([np.clip(np.nan_to_num(slope), 0, 60), np.log1p(np.clip(np.nan_to_num(hand), 0, 200))])
    return np.concatenate([sar, t]).astype('float32')
ids = [f.replace('_S1Hand.tif', '') for f in pd.read_csv(f'{ROOT}/flood_handlabeled/flood_train_data.csv', header=None)[0]]
xs = []
for c in ids:
    with rasterio.open(f'{ROOT}/S1Hand/{c}_S1Hand.tif') as s: sar = s.read().astype('float32')
    tt = np.load(f'{CACHE}/{c}.npy'); xs.append(prep(sar, tt[0], tt[1]))
xs = np.stack(xs); MEAN, STD = xs.mean((0, 2, 3)), xs.std((0, 2, 3)) + 1e-6; del xs
X = (prep(np.stack([b['flood_VV'], b['flood_VH']]), b['slope'], b['hand']) - MEAN[:, None, None]) / STD[:, None, None]
X[:, ~valid] = 0

@torch.no_grad()
def predict(model, x, tile=512, stride=448, batch=8):
    model.eval(); C_, H_, W_ = x.shape
    Hp = max(tile, math.ceil((H_ - tile) / stride) * stride + tile); Wp = max(tile, math.ceil((W_ - tile) / stride) * stride + tile)
    xp = np.pad(x, ((0, 0), (0, Hp - H_), (0, Wp - W_)), mode='edge')
    prob = np.zeros((Hp, Wp), 'float32'); cnt = np.zeros((Hp, Wp), 'float32')
    pos = [(r, c) for r in range(0, Hp - tile + 1, stride) for c in range(0, Wp - tile + 1, stride)]
    for i in range(0, len(pos), batch):
        pb = pos[i:i + batch]
        p = torch.softmax(model(torch.tensor(np.stack([xp[:, r:r+tile, c:c+tile] for r, c in pb])).to(DEV)), 1)[:, 1].cpu().numpy()
        for (r, c), pi in zip(pb, p): prob[r:r+tile, c:c+tile] += pi; cnt[r:r+tile, c:c+tile] += 1
    return (prob / cnt)[:H_, :W_] > 0.5

votes = np.zeros(shape, 'uint8')
for f in FT:
    m = smp.Unet('resnet34', encoder_weights=None, in_channels=4, classes=2).to(DEV)
    m.load_state_dict(torch.load(f, map_location=DEV)); votes += predict(m, X)
FLOOD = {'B fine-tuned (final)': (votes >= 2) & ~normal & valid,
         'threshold (VV < -15 dB)': (b['flood_VV'] < -15) & ~normal & valid}

z = glob.glob('/content/*FL20240928NPL*SHP*.zip') + glob.glob(f'{NEPAL}/*FL20240928NPL*SHP*.zip')
if z:
    import geopandas as gpd
    from rasterio.features import rasterize
    un = gpd.read_file(f'/vsizip/{z[0]}/FL20240928NPL_SHP/S1_20241001_FloodExtent_Koshi_Madhesh.shp').to_crs(prof['crs'])
    FLOOD['UNOSAT'] = rasterize(((g, 1) for g in un.geometry), out_shape=shape, transform=T, fill=0, dtype='uint8').astype(bool) & valid
else:
    print('UNOSAT zip not found - skipped')

with rasterio.open(f'{OUT}/koshi_new_water_B_finetuned.tif', 'w', **{**prof, 'count': 1, 'dtype': 'uint8', 'nodata': 255, 'compress': 'deflate'}) as dst:
    dst.write(np.where(valid, FLOOD['B fine-tuned (final)'], 255).astype('uint8'), 1)
for k, v in FLOOD.items(): print(f'{k}: {v.sum() / 1e4:.1f} km² new water')

## A2. Exposure layers from Earth Engine (exported once to Drive)
Bands: `bld` = buildings per 10 m pixel (Open Buildings v3 centroids, confidence ≥ 0.75), `pop` = WorldPop 2020 people per 10 m pixel, `district` = GAUL level-2 code.

In [ ]:
EXP = f'{NEPAL}/koshi_terai_exposure_layers.tif'
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filterBounds(AOI)
if not os.path.exists(EXP):
    bld = (ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons').filterBounds(AOI)
           .filter(ee.Filter.gte('confidence', 0.75))
           .map(lambda f: ee.Feature(f.geometry().centroid(1), {'one': 1})))
    bld_img = bld.reduceToImage(['one'], ee.Reducer.sum()).unmask(0).rename('bld')
    pop = (ee.ImageCollection('WorldPop/GP/100m/pop').filter(ee.Filter.eq('country', 'NPL'))
           .filter(ee.Filter.eq('year', 2020)).first().select('population').divide(100).unmask(0).rename('pop'))
    dist = gaul.reduceToImage(['ADM2_CODE'], ee.Reducer.first()).unmask(0).rename('district')
    img = ee.Image.cat([bld_img.reproject('EPSG:32645', None, 10), pop, dist]).toFloat().clip(AOI)
    task = ee.batch.Export.image.toDrive(image=img, description='koshi_exposure', folder='Paper2_flood_nepal',
            fileNamePrefix='koshi_terai_exposure_layers', region=AOI, scale=10, crs='EPSG:32645', maxPixels=1e10)
    task.start()
    while task.status()['state'] in ('READY', 'RUNNING'):
        print(task.status()['state']); time.sleep(30)
    print('export:', task.status()['state'], task.status().get('error_message', ''))
    for _ in range(20):
        if os.path.exists(EXP): break
        time.sleep(15)
assert os.path.exists(EXP), 'exposure file not in Drive yet - wait a minute and rerun this cell'

lay = np.zeros((3,) + shape, 'float32')
with rasterio.open(EXP) as src:
    for i in range(3):
        reproject(rasterio.band(src, i + 1), lay[i], dst_transform=T, dst_crs=prof['crs'], resampling=Resampling.nearest)
bld, pop, dist = lay[0], lay[1], lay[2].astype(int)
names = {int(f['properties']['ADM2_CODE']): f['properties']['ADM2_NAME'] for f in gaul.getInfo()['features']}
print(f'buildings in box: {bld[valid].sum():,.0f} | population: {pop[valid].sum():,.0f} | districts: {sorted(set(names.values()))}')
if bld[valid].sum() < 100: print('⚠️ almost no buildings found - Open Buildings may not cover this area; building numbers will be empty')

## A3. Roads from OpenStreetMap

In [ ]:
q = f'[out:json][timeout:180];way["highway"]({S},{W},{N},{E});out geom;'
r = requests.post('https://overpass-api.de/api/interpreter', data={'data': q}, timeout=300); r.raise_for_status()
ways = [w for w in r.json()['elements'] if 'geometry' in w]
MAJOR = {'motorway', 'trunk', 'primary', 'secondary', 'motorway_link', 'trunk_link', 'primary_link', 'secondary_link'}
tf = Transformer.from_crs('EPSG:4326', prof['crs'], always_xy=True)
STEP = 5.0
pts_r, pts_c, pts_major = [], [], []
for w in ways:
    lon = np.array([p['lon'] for p in w['geometry']]); lat = np.array([p['lat'] for p in w['geometry']])
    x, y = tf.transform(lon, lat)
    major = w.get('tags', {}).get('highway') in MAJOR
    for i in range(len(x) - 1):
        L = math.hypot(x[i+1] - x[i], y[i+1] - y[i]); n = max(int(L // STEP), 1)
        t_ = (np.arange(n) + 0.5) / n
        px, py = x[i] + t_ * (x[i+1] - x[i]), y[i] + t_ * (y[i+1] - y[i])
        col = ((px - T.c) / T.a).astype(int); row = ((py - T.f) / T.e).astype(int)
        ok = (row >= 0) & (row < shape[0]) & (col >= 0) & (col < shape[1])
        pts_r.append(row[ok]); pts_c.append(col[ok]); pts_major.append(np.full(ok.sum(), major))
pts_r, pts_c, pts_major = np.concatenate(pts_r), np.concatenate(pts_c), np.concatenate(pts_major)
pv = valid[pts_r, pts_c]
print(f'OSM ways: {len(ways):,} | road length in valid area: {pv.sum() * STEP / 1000:,.0f} km (major {(pv & pts_major).sum() * STEP / 1000:,.0f} km)')

## A4. Exposure tables
- **inside flood**: building centroid / road segment / population inside a new-water pixel.
- **within 100 m**: inside or within 100 m of new water. SAR misses water between buildings (double bounce), so "inside" underestimates exposure in villages; "within 100 m" is closer to what UNOSAT reports as "exposed or living near".

In [ ]:
near = lambda m: binary_dilation(m, iterations=10)
rows = []
for k, fm in FLOOD.items():
    nm = near(fm) & valid
    fr = fm[pts_r, pts_c]
    rows.append({'flood map': k, 'new water km²': round(fm.sum() / 1e4, 1),
                 'buildings inside': int(bld[fm].sum()), 'buildings within 100 m': int(bld[nm].sum()),
                 'people inside': int(pop[fm].sum()), 'people within 100 m': int(pop[nm].sum()),
                 'road km inside': round(fr.sum() * STEP / 1000, 1),
                 'major road km inside': round((fr & pts_major).sum() * STEP / 1000, 1)})
exposure = pd.DataFrame(rows); exposure.to_csv(f'{OUT}/exposure_by_method.csv', index=False); exposure

In [ ]:
fm = FLOOD['B fine-tuned (final)']; nm = near(fm) & valid
pd_ = dist[pts_r, pts_c]; fr = fm[pts_r, pts_c]
rows = []
for code_, name in sorted(names.items(), key=lambda kv: kv[1]):
    d = (dist == code_) & valid
    if d.sum() < 1e4: continue
    rows.append({'district': name, 'area in box km²': round(d.sum() / 1e4, 0),
                 'new water km²': round((fm & d).sum() / 1e4, 1),
                 'buildings within 100 m': int(bld[nm & d].sum()), 'people within 100 m': int(pop[nm & d].sum()),
                 'road km inside': round((fr & (pd_ == code_)).sum() * STEP / 1000, 1)})
by_district = pd.DataFrame(rows).sort_values('new water km²', ascending=False)
by_district.to_csv(f'{OUT}/exposure_by_district_B_finetuned.csv', index=False); by_district

# Part B – Paper figures
Colours: A blue, B orange, C aqua, D yellow (fixed order, colour-blind checked); every bar carries its value label.

In [ ]:
COL = {'A': '#2a78d6', 'B': '#eb6834', 'C': '#1baf7a', 'D': '#eda100'}
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({'font.size': 9, 'axes.edgecolor': INK2, 'axes.labelcolor': INK, 'xtick.color': INK2,
                     'ytick.color': INK2, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': GRID, 'grid.linewidth': 0.6,
                     'axes.axisbelow': True, 'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})
FIG = f'{OUT}/figures'; os.makedirs(FIG, exist_ok=True)

def bar_labels(ax, bars, vals, fmt='{:.3f}', dy=0.004):
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + dy, fmt.format(v),
                ha='center', va='bottom', fontsize=7.5, color=INK)

## Figure 1 – Benchmark: IoU on flat vs sloped pixels (Sen1Floods11 test, 3 seeds)

In [ ]:
runs = pd.concat([pd.read_csv(f) for f in glob.glob(f'{BASE}/05_multiseed/runs/*.csv') + glob.glob(f'{BASE}/07_attention/runs/*.csv')])
runs = runs[(runs.split == 'test') & runs.terrain.isin(['flat <5°', 'sloped ≥5°'])]
agg = runs.groupby(['model', 'terrain'])['IoU'].agg(['mean', 'std'])
models = [('A_sar', 'A', 'A: U-Net'), ('B_sar_terrain', 'B', 'B: U-Net + terrain'),
          ('C_att_sar', 'C', 'C: attention'), ('D_att_sar_terrain', 'D', 'D: attention + terrain')]
fig, ax = plt.subplots(figsize=(6.5, 3.4))
groups = ['flat <5°', 'sloped ≥5°']; w = 0.19
for i, (m, k, lab) in enumerate(models):
    mu = [agg.loc[(m, g), 'mean'] for g in groups]; sd = [agg.loc[(m, g), 'std'] for g in groups]
    xs_ = np.arange(2) + (i - 1.5) * (w + 0.015)
    bars = ax.bar(xs_, mu, w, color=COL[k], label=lab, yerr=sd, ecolor=INK2, capsize=2, error_kw={'linewidth': 0.8})
    bar_labels(ax, bars, mu, dy=max(sd) + 0.006)
ax.set_xticks(range(2)); ax.set_xticklabels(['Flat pixels (slope < 5°)', 'Sloped pixels (slope ≥ 5°)'])
ax.set_ylabel('Water IoU'); ax.set_ylim(0, 0.8)
ax.legend(ncol=4, frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.13))
ax.set_title('Accuracy drops on slopes; terrain inputs help most there', loc='left', fontsize=10, color=INK)
plt.savefig(f'{FIG}/fig1_benchmark_flat_vs_sloped.png'); plt.show()

## Figure 2 – Kathmandu false alarms (flood already receded on 2 Oct)

In [ ]:
ft = pd.read_csv(f'{BASE}/08_finetune/finetune_results_per_seed.csv')
s04 = pd.read_csv(f'{BASE}/04_nepal_inference/nepal_summary.csv')
cd = s04[(s04.area == 'kathmandu') & (s04.method == 'quick change detection')].iloc[0]
items = [('Change detection', cd['new water km²'], 0, cd['new in layover/shadow km²'], INK2)]
for name, k in [('A_sar', 'A'), ('B_sar_terrain', 'B')]:
    for ver in ['original', 'fine-tuned']:
        d = ft[(ft.model == name) & (ft.version == ver)]
        items.append((f'{k} {ver}', d['KTM false new water km²'].mean(), d['KTM false new water km²'].std(),
                      d['KTM in layover/shadow km²'].mean(), COL[k]))
fig, ax = plt.subplots(figsize=(6.5, 3.0))
ys = np.arange(len(items))[::-1]
for y, (lab, v, sd, ls, c) in zip(ys, items):
    ax.barh(y, v, 0.6, color=c, xerr=sd if sd > 0 else None, ecolor=INK2, capsize=2, error_kw={'linewidth': 0.8})
    ax.text(v + (sd if sd > 0 else 0) + 0.12, y, f'{v:.2f} km²  (layover/shadow {ls:.2f})', va='center', fontsize=7.5, color=INK)
ax.set_yticks(ys); ax.set_yticklabels([i[0] for i in items])
ax.grid(axis='x', color=GRID); ax.grid(axis='y', visible=False)
ax.set_xlabel('False new water in Kathmandu scene (km², ~630 km² area)'); ax.set_xlim(0, max(i[1] + i[2] for i in items) * 1.55)
ax.set_title('U-Nets cut false alarms; terrain inputs remove layover/shadow errors', loc='left', fontsize=10, color=INK)
plt.savefig(f'{FIG}/fig2_kathmandu_false_alarms.png'); plt.show()

## Figure 3 – Fine-tuning on Nepal data (Koshi south half, new-water IoU vs Sentinel-2)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.2)); w = 0.34
for i, (name, k) in enumerate([('A_sar', 'A'), ('B_sar_terrain', 'B')]):
    for j, ver in enumerate(['original', 'fine-tuned']):
        d = ft[(ft.model == name) & (ft.version == ver)]['L2 new-water IoU']
        x = i + (j - 0.5) * (w + 0.03)
        bar = ax.bar(x, d.mean(), w, color=COL[k], alpha=0.45 if ver == 'original' else 1.0,
                     yerr=d.std(), ecolor=INK2, capsize=2, error_kw={'linewidth': 0.8})
        bar_labels(ax, bar, [d.mean()], dy=d.std() + 0.006)
        ax.text(x, 0.015, ver, ha='center', va='bottom', fontsize=7, color=INK, rotation=90)
thr = 0.407
ax.axhline(thr, color=INK2, linewidth=1, linestyle='--', label=f'threshold method ({thr:.3f})')
ax.legend(frameon=False, loc='upper left', fontsize=8)
ax.set_xticks([0, 1]); ax.set_xticklabels(['A: U-Net', 'B: U-Net + terrain']); ax.set_xlim(-0.6, 1.6)
ax.set_ylabel('New-water IoU'); ax.set_ylim(0, 0.6)
ax.set_title('Fine-tuning on one Nepal image pair improves both models', loc='left', fontsize=10, color=INK)
plt.savefig(f'{FIG}/fig3_finetuning_koshi.png'); plt.show()

## Figure 4 – Koshi maps: SAR, final flood map with exposed buildings, agreement with Sentinel-2

In [ ]:
with rasterio.open(LAB8) as src:
    lab = np.full((2,) + shape, 255, 'float32')
    for i in range(2):
        reproject(rasterio.band(src, i + 1), lab[i], dst_transform=T, dst_crs=prof['crs'], resampling=Resampling.nearest)
m2 = valid & (lab[0] != 255) & (lab[1] != 255); truth = (lab[0] == 1) & (lab[1] == 0)
fm = FLOOD['B fine-tuned (final)']; st = 3

fig, ax = plt.subplots(1, 3, figsize=(13, 4.8))
ax[0].imshow(np.where(valid, np.clip(b['flood_VV'], -25, 0), np.nan)[::st, ::st], cmap='gray')
ax[0].set_title('(a) Sentinel-1 VV, 1 Oct 2024', loc='left', fontsize=10)

im = np.ones(shape + (3,)); im[~valid] = 0.92
im[normal & valid] = [0.70, 0.78, 0.88]; im[fm] = [0.16, 0.47, 0.84]
exp_b = (bld > 0) & near(fm) & valid
im[binary_dilation(exp_b, iterations=2)] = [0.89, 0.29, 0.28]
ax[1].imshow(im[::st, ::st])
ax[1].set_title('(b) New water (blue), normal water (light blue),\n     buildings within 100 m (red)', loc='left', fontsize=10)

im2 = np.full(shape + (3,), 0.85); im2[m2] = 1
im2[m2 & fm & truth] = [0.11, 0.69, 0.48]; im2[m2 & fm & ~truth] = [0.89, 0.29, 0.28]; im2[m2 & ~fm & truth] = [0.16, 0.47, 0.84]
ax[2].imshow(im2[::st, ::st])
ax[2].set_title('(c) vs Sentinel-2, 8 Oct: correct (green),\n     false (red), missed (blue), no label (grey)', loc='left', fontsize=10)
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig(f'{FIG}/fig4_koshi_maps.png'); plt.show()
print('figures saved to', FIG)